##### ============================================================
### repex_topology_parser
#### Currently only supports Amber Potential 
#### without CMAP corrections
$$
E_{\rm total} = \sum_{\rm bonds} K_r (r - r_{\rm eq})^2 
+ \sum_{\rm angles} K_\theta (\theta - \theta_{\rm eq})^2 
+ \sum_{\rm dihedrals} \frac{V_n}{2} \left[1 + \cos(n\phi - \gamma)\right] 
+ \sum_{i<j} \epsilon_{ij} \left[ \left( \frac{\sigma_{ij}}{R_{ij}} \right)^{12} - \left( \frac{\sigma_{ij}}{R_{ij}} \right)^6 \right] 
+ \sum_{i<j} \frac{q_i q_j}{4 \pi \epsilon_0 R_{ij}}
$$
### TODO
#### - Add CMAP lambda scaling
#### - Extend to CHARMM, OPLS-AA
### Examples for REST2, and ssREST3 (solvent-scaled REST2)
##### ============================================================

# ===========================================
### Energy Components
$$ E_{\rm total} = E_{\rm protein-protein} + E_{\rm protein-water} + E_{\rm water-water} $$
#### REST2 scaling with $\lambda$
$$ \lambda = \frac{\beta_n}{\beta_0} = \frac{T_0}{T_n} : \beta_n = \frac{1}{k_B T_n} $$
$$ E_{\rm total}^{\lambda_n} = \lambda_n E_{\rm protein-protein} + \sqrt{\lambda_n} E_{\rm protein-water} + E_{\rm water-water} $$
### solvent-scaling 
$$ \kappa_i =  e^{\frac{i}{N - 1} log(\kappa_{max})} ; i \in [0,N-1]$$
$$ E_{\rm total}^{\kappa_n} = E_{\rm protein-protein} + \kappa_n E_{\rm protein-water} + E_{\rm water-water} $$
### ssREST2 (solvent-scaled REST2)
$$ E_{\rm total}^{\lambda_n,\kappa_n} = \lambda_nE_{\rm protein-protein} + \boldsymbol{\kappa_n} \sqrt{\lambda_n} E_{\rm protein-water} + E_{\rm water-water} $$

# ===========================================

In [1]:
# Load our library
import src.repex_topology_parser as rtp
from pathlib import Path
Path("./example_conversion").mkdir(parents=True, exist_ok=True)

In [2]:
# initialize our class providing an input processed.top file
example_scaling = rtp.topo2rest('/home/koreyr/data/FUS/apoFUS_8.5nm_4mer_wsrex/wsrex_setup/processed.top',nreps=4)

In [3]:
### Let us display our molecules contained in our topology
example_scaling.molecules

{0: 'Protein_chain_A',
 1: 'Protein_chain_B',
 2: 'Protein_chain_C',
 3: 'Protein_chain_D',
 4: 'SOL',
 5: 'NA',
 6: 'CL'}

In [4]:
example_scaling.show_molecule_atomtypes(1)

0: C
1: C1
2: C5
3: C6
4: C7
5: C8
6: C9
7: CA
8: CT
9: H
10: H1
11: HA
12: HB
13: HC
14: HO
15: HP
16: N
17: N3
18: O
19: O2
20: O3
21: OB
22: OH
23: S


### Here we perform with one command rest2 scaling. The hot molecule will have dihedrals/charges/LJ 
### parameters scaled by $\sqrt\lambda$. Thus intra-hot molecule interactions will be scaled
### by $\lambda$, while hot molecule - other molecules will be scaled by $\sqrt\lambda$

In [5]:
example_scaling._sections['moleculetype'][0]

{'header': ['[ moleculetype ]\n',
  '; Name            nrexcl\n',
  'Protein_chain_A     3\n'],
 'atoms': ['     1         N3      1    GLY      N      1     0.2943      14.01\n',
  '     2          H      1    GLY     H1      2     0.1642      1.008\n',
  '     3          H      1    GLY     H2      3     0.1642      1.008\n',
  '     4          H      1    GLY     H3      4     0.1642      1.008\n',
  '     5         C1      1    GLY     CA      5      -0.01      12.01\n',
  '     6         HP      1    GLY    HA1      6     0.0895      1.008\n',
  '     7         HP      1    GLY    HA2      7     0.0895      1.008\n',
  '     8          C      1    GLY      C      8     0.6163      12.01\n',
  '     9         OB      1    GLY      O      9    -0.5722         16   ; qtot 1\n',
  '    10          N      2    MET      N     10    -0.4157      14.01\n',
  '    11         HB      2    MET      H     11     0.2719      1.008\n',
  '    12         CT      2    MET     CA     12    -0.0237

In [6]:
# Our first example is performing solute scaling (REST2) on just the protein
REST2 = { 'hot_molecules':[0], # Select the molecule(s) you desire to scale, as a list
            'nreps':20, # define the number of replicas you desire
            'outfile':'topol_rest2_apo_sys2', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'/home/koreyr/bigData/check_michelle/dimer_apo/processed.top', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'rest2', # method is rest2
            'temps':[300,400], # temperature range 300 to 500, utilized to compute the geometric
                               # temperature ladder
                               # default = [300,500]
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
example_scaling.run(**REST2)

Running rest2 scaling method
[(2, 1, 5, 6, 9), (2, 1, 5, 7, 9), (2, 1, 5, 23, 9), (3, 1, 5, 6, 9), (3, 1, 5, 7, 9), (3, 1, 5, 23, 9), (4, 1, 5, 6, 9), (4, 1, 5, 7, 9), (4, 1, 5, 23, 9), (1, 5, 7, 8, 9), (1, 5, 7, 9, 9), (1, 5, 7, 10, 9), (6, 5, 7, 8, 9), (6, 5, 7, 9, 9), (6, 5, 7, 10, 9), (23, 5, 7, 8, 9), (23, 5, 7, 9, 9), (23, 5, 7, 10, 9), (1, 5, 23, 24, 9), (1, 5, 23, 25, 9), (6, 5, 23, 24, 9), (6, 5, 23, 25, 9), (7, 5, 23, 24, 9), (7, 5, 23, 25, 9), (5, 7, 10, 11, 9), (5, 7, 10, 12, 9), (5, 7, 10, 13, 9), (8, 7, 10, 11, 9), (8, 7, 10, 12, 9), (8, 7, 10, 13, 9), (9, 7, 10, 11, 9), (9, 7, 10, 12, 9), (9, 7, 10, 13, 9), (7, 10, 13, 14, 9), (7, 10, 13, 15, 9), (7, 10, 13, 16, 9), (11, 10, 13, 14, 9), (11, 10, 13, 15, 9), (11, 10, 13, 16, 9), (12, 10, 13, 14, 9), (12, 10, 13, 15, 9), (12, 10, 13, 16, 9), (10, 13, 16, 17, 9), (10, 13, 16, 18, 9), (10, 13, 16, 19, 9), (14, 13, 16, 17, 9), (14, 13, 16, 18, 9), (14, 13, 16, 19, 9), (15, 13, 16, 17, 9), (15, 13, 16, 18, 9), (15, 13, 16, 19,

### Here we perform ssrest3 scaling with one command, ssrest3 applies additional scaling on the OW atom of our water model, in this case \'OW_tip4pd\' atomtype. 
### First, the input of a hot molecule has the same effect as rest2 where the protein dihedrals/charges/LJ parameters are scaled by $\lambda$.
### Second the LJ $\epsilon$ of water is scaled by $\kappa^2$ tuning the solvation of the hot molecule(s).
### And lastly, all non-hot molecule-water LJ parameters are reset thus avoiding the effects of solvent scaling. 

In [11]:
# Our second example is performing solvent-scaling REST3 (ssREST3) on just the protein
# From the displayed atomtypes contained in our solvent molecule we opt to scale
# 'OW_tip4pd' ('HW' and 'MW' have epsilon = 0.0 so we exclude these from the list)
ssREST3 = { 'hot_molecules':[0,1,2,3], # Select the molecule(s) you desire to scale, as a list
            'nreps':4, # define the number of replicas you desire
            'outfile':'topol_ssrest3_apo_sys2', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'/home/koreyr/files_for_Kaushik/SJ4/', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'ssrest3', # method is ssrest3
            'temps':[300,500], # temperature range 300 to 500, utilized to compute the geometric
                               # temperature ladder
                               # default = [300,500]
            'kappa_low_temp' : 300, # At which temperature to activate solvent scaling, maybe useful
                                    # to increse to 330 when using a lower number of replicas so the base replica
                                    # experiences solvation more accurately. For ssREST3 simulations with 
                                    # 16 or more replicas, it is unlikedly changing this value to 330 will have 
                                    # any benefit. 
            'kappa_max' : 1.07, # Maximum kappa value
            'kappa_atom_names' : ['OW_tip4pd'], # List of atomtypes to apply kappa scaling
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
example_scaling.run(**ssREST3)

Running ssrest3 scaling method
[(2, 1, 5, 6, 9), (2, 1, 5, 7, 9), (2, 1, 5, 8, 9), (3, 1, 5, 6, 9), (3, 1, 5, 7, 9), (3, 1, 5, 8, 9), (4, 1, 5, 6, 9), (4, 1, 5, 7, 9), (4, 1, 5, 8, 9), (1, 5, 8, 9, 9), (1, 5, 8, 10, 9), (6, 5, 8, 9, 9), (6, 5, 8, 10, 9), (7, 5, 8, 9, 9), (7, 5, 8, 10, 9), (5, 8, 10, 11, 9), (5, 8, 10, 12, 9), (9, 8, 10, 11, 9), (9, 8, 10, 12, 9), (8, 10, 12, 13, 9), (8, 10, 12, 14, 9), (8, 10, 12, 25, 9), (11, 10, 12, 13, 9), (11, 10, 12, 14, 9), (11, 10, 12, 25, 9), (10, 12, 14, 15, 9), (10, 12, 14, 16, 9), (10, 12, 14, 17, 9), (13, 12, 14, 15, 9), (13, 12, 14, 16, 9), (13, 12, 14, 17, 9), (25, 12, 14, 15, 9), (25, 12, 14, 16, 9), (25, 12, 14, 17, 9), (10, 12, 25, 26, 9), (10, 12, 25, 27, 9), (13, 12, 25, 26, 9), (13, 12, 25, 27, 9), (14, 12, 25, 26, 9), (14, 12, 25, 27, 9), (12, 14, 17, 18, 9), (12, 14, 17, 19, 9), (12, 14, 17, 20, 9), (15, 14, 17, 18, 9), (15, 14, 17, 19, 9), (15, 14, 17, 20, 9), (16, 14, 17, 18, 9), (16, 14, 17, 19, 9), (16, 14, 17, 20, 9), (14, 17

##### The most prudent approach is to perform time continuous simulations of the hot or hightemp replica applying different solvent scaling. 
##### In this example the hot replica's effective temperature is 500 K.
##### The conformational sampling of each hightemp replica's continuous simulation should be compared to a continuous unscaled simulation performed with a bath temperature matching the selected effective temperature of the hightemp replica; 200 to 500 ns should do, however longer simulations may be needed.
##### For fully disorderd IDPs: when tested on the 20-residue fragment of $\alpha$-synuclien 200 ns was more than reasonable, a 40-residue fragment of Fused in Sarcoma ~500 ns, and in contrast ~1 $\mu$s for a 70-residue fragment of p27.
##### For this paper, simple metrics were introduced to assess the appropriateness of $\kappa_{max}$ such as radius of gyration and secondary structure population, you are invited to test other metrics that are important to your study. 
#####
##### To minimize the real world time required to sample $\kappa_{max}$ values we opted to select 9 values of $\kappa_{max}$ to test with a $\lambda_{max}$ value of 0.6 corresponding to an effective temperature of 500 K. 
##### On a 10 GPU system this results in:
##### -- One unscaled simulation with a bath temperature coupled to 500 K
##### -- 9 scaled simulations with $\kappa_{max}$ ranging from 1.02 to 1.1
##### Gromacs 2022.5 was compiled with mpi and cuda support. 
##### Simulations were all submitted together using the mdrun flag -multidir:
```bash
gmx_gpu_mpi mdrun -v -deffnm run -mutlidir 500K kappa{1..9}
```

#### The code below generatures the unscaled topology and the 9 values of $\kappa_{max}$:
##### 1.02, 1.03, 1.04, 1.05, 1.06, 1.07, 1.08, 1.09 and 1.1.
#### It is important to check the base replica topology against the processed topology.

In [14]:
from numpy import arange
for k in arange(1.03,1.05,0.01):
    for T in [400,450,500]:
        for reps in [10,20]:
            kappa_scan = rtp.topo2rest('/home/koreyr/bigData/check_michelle/dimer_apo/processed.top', nreps=reps, temps=[300,T])
            Path(f"/home/koreyr/bigData/check_michelle/dimer_apo/{T}K_reps{reps}_kappa{k}").mkdir(parents=True, exist_ok=True)
            ssREST3 = { 'hot_molecules':[0], 
                'nreps':reps, 
                'outfile':f'topol_ssrest3',          
                'filepath':f'/home/koreyr/bigData/check_michelle/dimer_apo/{T}K_reps{reps}_kappa{k}/', 
                'method':'ssrest3',
                'temps':[300,T], 
                'kappa_low_temp' : 300, 
                'kappa_max' : k, 
                'kappa_atom_names' : ['OW_tip4pd'], 
                'verbose':True     
                }
            kappa_scan.run(**ssREST3)

Running ssrest3 scaling method
[(2, 1, 5, 6, 9), (2, 1, 5, 7, 9), (2, 1, 5, 23, 9), (3, 1, 5, 6, 9), (3, 1, 5, 7, 9), (3, 1, 5, 23, 9), (4, 1, 5, 6, 9), (4, 1, 5, 7, 9), (4, 1, 5, 23, 9), (1, 5, 7, 8, 9), (1, 5, 7, 9, 9), (1, 5, 7, 10, 9), (6, 5, 7, 8, 9), (6, 5, 7, 9, 9), (6, 5, 7, 10, 9), (23, 5, 7, 8, 9), (23, 5, 7, 9, 9), (23, 5, 7, 10, 9), (1, 5, 23, 24, 9), (1, 5, 23, 25, 9), (6, 5, 23, 24, 9), (6, 5, 23, 25, 9), (7, 5, 23, 24, 9), (7, 5, 23, 25, 9), (5, 7, 10, 11, 9), (5, 7, 10, 12, 9), (5, 7, 10, 13, 9), (8, 7, 10, 11, 9), (8, 7, 10, 12, 9), (8, 7, 10, 13, 9), (9, 7, 10, 11, 9), (9, 7, 10, 12, 9), (9, 7, 10, 13, 9), (7, 10, 13, 14, 9), (7, 10, 13, 15, 9), (7, 10, 13, 16, 9), (11, 10, 13, 14, 9), (11, 10, 13, 15, 9), (11, 10, 13, 16, 9), (12, 10, 13, 14, 9), (12, 10, 13, 15, 9), (12, 10, 13, 16, 9), (10, 13, 16, 17, 9), (10, 13, 16, 18, 9), (10, 13, 16, 19, 9), (14, 13, 16, 17, 9), (14, 13, 16, 18, 9), (14, 13, 16, 19, 9), (15, 13, 16, 17, 9), (15, 13, 16, 18, 9), (15, 13, 16, 1

In [6]:
wsrepex = { 'hot_molecules':[0,1,2,3], # Select the molecule(s) you desire to scale, as a list
            'nreps':4, # define the number of replicas you desire
            'outfile':'topol_wsrepex', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'/home/koreyr/data/FUS/apoFUS_8.5nm_4mer_wsrex/wsrex_setup/', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'solvent_scaled', # method is ssrest3
            'kappa_low_temp' : 300, # At which temperature to activate solvent scaling, maybe useful
                                    # to increse to 330 when using a lower number of replicas so the base replica
                                    # experiences solvation more accurately. For ssREST3 simulations with 
                                    # 16 or more replicas, it is unlikedly changing this value to 330 will have 
                                    # any benefit. 
            'kappa_max' : 1.07, # Maximum kappa value
            'kappa_atom_names' : ['OW_tip4pd'], # List of atomtypes to apply kappa scaling
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
example_scaling.run(**wsrepex)


solvent_scaled
Running solvent scaling method


In [7]:
example_scaling.molecules

{0: 'Protein_chain_A',
 1: 'Protein_chain_B',
 2: 'Protein_chain_C',
 3: 'Protein_chain_D',
 4: 'SOL',
 5: 'NA',
 6: 'CL'}

In [8]:
example_scaling._sections

{'defaults': [' [ defaults ] \n',
  '; nbfunc        comb-rule       gen-pairs       fudgeLJ fudgeQQ\n',
  '1               2               yes             0.5     0.8333\n'],
 'atomtypes': [' [ atomtypes ] \n',
  '; name      at.num  mass     charge ptype  sigma      epsilon\n',
  'Br          35      79.90    0.0000  A   0.00000e+00  0.00000e+00\n',
  'C            6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'C6           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'C5           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'CA           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'CB           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'CC           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'CK           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'CM           6      12.01    0.0000  A   3.39967e-01  3.59824e-01\n',
  'CN           6      12.01    0.0000  A   3.39967e-01  3.598

### Scaling can also be performed from the command line, either you supply the processed topology, the method, the temperature range and if desired $\kappa_{max}$ and enter into an interactive session to select the hot molecule(s) and the solvent to be scaled. 
# Or
### All options can be provided from the command line as shown below. 

```bash
username@computer] python3 src/repex_topology_parser.py --help
usage: repex_topology_parser [-h] [-p TOPOL] [--show-molecules] [--show-molecule-atomtypes SHOW_MOLECULE_ATOMTYPES] [--verbose] [-O OUTFILE] [-P OUTPUT]
                             [-m METHOD] [-H HOT_MOLECULES [HOT_MOLECULES ...]] [-n NREPS] [-k KAPPA_MAX] [--kappa-low-temp KAPPA_LOW_TEMP]
                             [--kappa-atomtypes KAPPA_ATOMTYPES [KAPPA_ATOMTYPES ...]] [-T TMIN] [-M TMAX]

Parse GROMACS processed topology file and perform desired scaling.

optional arguments:
  -h, --help            show this help message and exit
  -p TOPOL, --topol TOPOL
                        Processed Topology (default: processed.top)
  --show-molecules      Show molecule indices found in processed.top
  --show-molecule-atomtypes SHOW_MOLECULE_ATOMTYPES
                        Show molecule atomtypes found in processed.top
  --verbose
  -O OUTFILE, --outfile OUTFILE
                        Scaled topology base name (default: topol)
  -P OUTPUT, --output OUTPUT
                        Scaled topology output PATH (default: ./)
  -m METHOD, --method METHOD
                        Scaled topology method [ rest2 | ssrest3 ] (default: rest2)
  -H HOT_MOLECULES [HOT_MOLECULES ...], --hot-molecules HOT_MOLECULES [HOT_MOLECULES ...]
                        Space delimited entry of hot molecule selection (default: [0])
  -n NREPS, --nreps NREPS
                        Number of replicas (default: 10)
  -k KAPPA_MAX, --kappa-max KAPPA_MAX
                        Max Kappa Scaling (default: 1.06)
  --kappa-low-temp KAPPA_LOW_TEMP
                        Replicas at or above this temperature will have active solvent scaling (default:300)
  --kappa-atomtypes KAPPA_ATOMTYPES [KAPPA_ATOMTYPES ...]
                        Atomtypes to apply solvent scaling (default:['OW'])
  -T TMIN, --tmin TMIN  Base Temperature (default: 300)
  -M TMAX, --tmax TMAX  Max Temperature (default: 500)

Happy scaling!
```

### First query the molecules in the topology
```bash
username@computer] python3 src/repex_topology_parser.py -p tests/topology_files/test_topo/processed.top --show-molecules
0: Protein_chain_A
1: HxD
2: SOL
3: NA
4: CL
```

### Scaling with REST2 selecting 'Protein_chain_A' is simple:
```bash
username@computer] python3 src/repex_topology_parser.py -p tests/topology_files/test_topo/processed.top -p tests/topology_files/test_topo/processed.top -P tests/test_cli/ -m rest2 -H 0 -n 20 -O topol_rest2 --verbose
topology file read
Running rest2 scaling method
```

# For ssREST3, first to identify the atom(s) that will be scaled. For the a99disp-water model it is 
# important to note only the Oxygen heavy atom contains a non-zero LJ $\epsilon$ value and solvent scaling
# only applies to the LJ $\epsilon$ term. Therefore only the associated Oxygen atomtype is selected.
```bash
username@computer] python3 src/repex_topology_parser.py -p tests/topology_files/test_topo/processed.top --show-molecule-atomtypes 2
0: HW
1: MW
2: OW_tip4pd

# The command line ssREST3 scaling providing all options to cercumvent interactive mode:
```bash
username@computer] python3 src/repex_topology_parser.py -p tests/topology_files/test_topo/processed.top -p tests/topology_files/test_topo/processed.top -P tests/test_cli/ -m ssrest3 -H 0 -n 20 -k 1.1 --kappa-atomtypes OW_tip4pd -O topol_ssrest3 --verbose
topology file read
Running ssrest3 scaling method
```

### With interactive mode you can simply provide the topology, recognize the default values applied and 
### answer the questionaires to produce scaled topologies, however, this is discouraged for production runs. 
### Lastly, poor exchange ratios between specific replicas may occur and from within a jupyter notebook you 
### can supply the 'temps_opt':[ <temps(float)> ] to fine tune the temperature ladder. 